# AMR Parsing Inference Pipeline

This notebook demonstrates how to use the fine-tuned AMRBART model to parse sentences into AMR (Abstract Meaning Representation) graphs.

**Model**: `mbart-en-id-smaller-concat-finetuned`  
**Task**: Text → AMR (Sentence to AMR graph)

## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import torch
import penman
import numpy as np

# Absolute path to the repository modules. Change this line if the project moves.
MODULES_PATH = r"D:\Github\generate_amr"
if MODULES_PATH not in sys.path:
    sys.path.insert(0, MODULES_PATH)

from transformers import AutoConfig
from model_interface.modeling_bart import MBartForConditionalGeneration
from model_interface.tokenization_bart import AMRBartTokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Load Model & Tokenizer

In [ ]:
# Absolute path to the extracted fine-tuned model.
MODEL_PATH = r"D:\Github\generate_amr\models\mbart-en-id-smaller-concat-finetuned\mbart-en-id-smaller-concat-finetuned"

# Alternative model:
# MODEL_PATH = r"D:\Github\generate_amr\models\mbart-en-id-smaller-concat-trans-finetuned\mbart-en-id-smaller-concat-trans-finetuned"

print(f"Loading model from: {MODEL_PATH}")
print(f"Model files: {os.listdir(MODEL_PATH)}")

In [ ]:
# Load config
config = AutoConfig.from_pretrained(MODEL_PATH)
print(f"Model type: {config.model_type}")
print(f"Architecture: {config.architectures}")
print(f"Vocab size: {config.vocab_size}")

In [ ]:
# Load tokenizer (custom AMRBartTokenizer)
tokenizer = AMRBartTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"AMR BOS token: {tokenizer.amr_bos_token} (id={tokenizer.amr_bos_token_id})")
print(f"AMR EOS token: {tokenizer.amr_eos_token} (id={tokenizer.amr_eos_token_id})")

In [ ]:
# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MBartForConditionalGeneration.from_pretrained(MODEL_PATH, config=config)
model.resize_token_embeddings(len(tokenizer))
model = model.to(device)
model.eval()

print(f"Model loaded on: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Load Example Data

We'll load a few examples from `val.jsonl` and `test.jsonl` to test the model.

In [ ]:
# Absolute paths to the example datasets.
VAL_DATA_PATH = r"D:\Github\generate_amr\examples\val.jsonl"
TEST_DATA_PATH = r"D:\Github\generate_amr\examples\test.jsonl"

val_data = []
with open(VAL_DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            val_data.append(json.loads(line))

test_data = []
with open(TEST_DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            test_data.append(json.loads(line))

print(f"Loaded {len(val_data)} validation examples")
print(f"Loaded {len(test_data)} test examples")
print(f"\nExample val sentence: {val_data[0]['sent'][:100]}...")
print(f"Example test sentence: {test_data[0]['sent'][:100]}...")

## 4. Define AMR Parsing Function

This function takes a sentence and produces an AMR graph using the model.

The input format for this "concat" model follows the pattern used during training:
- The sentence is tokenized with a language prefix (e.g., `en_XX` for English)
- Special AMR tokens (`<AMR>`, `<mask>`, `</AMR>`) are appended as a unified input signal

In [ ]:
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM
from huggingface_hub import snapshot_download

# Download model locally first to bypass transformers/huggingface_hub version mismatch
nllb_local_path = snapshot_download("facebook/nllb-200-distilled-600M")
print(f"NLLB model downloaded to: {nllb_local_path}")

nllb_tokenizer = NllbTokenizer.from_pretrained(nllb_local_path)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_local_path)
print(f"NLLB model loaded successfully!")
nllb_model.device

In [ ]:
article = "Saya tidak mengerti maksud anda"
inputs = nllb_tokenizer(article, return_tensors="pt").to(nllb_model.device)

translated_tokens = nllb_model.generate(
    **inputs, forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("eng_Latn"), max_length=30
)
nllb_tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

In [ ]:

def parse_sentence_to_amr(
    sentence: str,
    model,
    tokenizer,
    device,
    lang: str = "en_XX",
    max_src_length: int = 400,
    max_tgt_length: int = 1024,
    num_beams: int = 5,
    is_concat : bool = True
):
    """
    Parse a sentence into an AMR graph.

    Args:
        sentence: Input sentence to parse
        model: The fine-tuned MBart model
        tokenizer: The AMRBartTokenizer
        device: torch device
        lang: Language code prefix ("en_XX" for English, "id_ID" for Indonesian)
        max_src_length: Maximum source sequence length
        max_tgt_length: Maximum target (AMR) sequence length
        num_beams: Number of beams for beam search

    Returns:
        dict with 'amr_graph' (penman Graph), 'amr_string' (str), 'status' (ParsedStatus)
    """
    # Tokenize with unified input format:
    # [lang_prefix + sentence tokens ... <AMR> <mask> </AMR>]
    text = lang + sentence

    if is_concat:
        tes = "I don't understand what you mean"
        text += "en_XX" + tes

    raw_ids = tokenizer(
        text, max_length=max_src_length, padding=False, truncation=True
    )["input_ids"]

    # Append AMR special tokens for unified input
    input_ids = (
        raw_ids[: max_src_length - 3]
        + [tokenizer.amr_bos_token_id, tokenizer.mask_token_id, tokenizer.amr_eos_token_id]
    )

    # Convert to tensor and add batch dimension
    input_ids_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    # Generate AMR tokens
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids_tensor,
            max_length=max_tgt_length,
            num_beams=num_beams,
            decoder_start_token_id=tokenizer.amr_bos_token_id,
        )

    # Decode the generated AMR tokens
    pred_ids = outputs[0].cpu().tolist()

    # Fix: ensure first token is bos
    pred_ids[0] = tokenizer.bos_token_id

    # Replace amr_eos with eos and remove padding
    pred_ids = [
        tokenizer.eos_token_id if tok == tokenizer.amr_eos_token_id else tok
        for tok in pred_ids
        if tok != tokenizer.pad_token_id
    ]

    # Decode into AMR graph
    graph, status, (nodes, backreferences) = tokenizer.decode_amr(
        pred_ids, restore_name_ops=False
    )

    # Add metadata
    if hasattr(graph, 'metadata'):
        graph.metadata["snt"] = sentence
    else:
        graph.metadata = {"snt": sentence}

    # Encode the graph into a readable string
    amr_string = penman.encode(graph)

    return {
        "amr_graph": graph,
        "amr_string": amr_string,
        "status": status,
        "raw_pred_ids": pred_ids,
    }

## 5. Test: Parse a Single Sentence

Let's pick one example from `test.jsonl` and parse it.

In [ ]:
# Pick a simple example from test data
example = test_data[0]
# sentence = example["sent"]
sentence = "saya tidak mengerti maksud anda"
gold_amr = example["amr"]

print(f"Input sentence: {sentence}")
print(f"\nGold AMR (linearized): {gold_amr}")

In [ ]:
# Run inference
result = parse_sentence_to_amr(sentence, model, tokenizer, device, lang="id_ID", is_concat=True)

print(f"Parsing status: {result['status']}")
print(f"\n{'='*60}")
print(f"PREDICTED AMR GRAPH:")
print(f"{'='*60}")
print(result["amr_string"])

## 6. Test: Parse Multiple Sentences

Let's test with several examples from both val and test sets.

In [ ]:
# Select a mix of examples
examples_to_test = [
    ("test", test_data[4]),   # "September 11th, 2010"
    ("test", test_data[7]),   # "How Long are We Going to Tolerate Japan?"
    ("test", test_data[8]),   # "My fellow citizens:"
    ("val", val_data[2]),     # "taking a look"
    ("val", val_data[9]),     # "Just passing by and taking a look..."
]

print(f"Testing {len(examples_to_test)} sentences...\n")

for i, (split, example) in enumerate(examples_to_test):
    sentence = example["sent"]
    gold_amr = example["amr"]

    print(f"{'='*80}")
    print(f"Example {i+1} ({split})")
    print(f"{'='*80}")
    print(f"Sentence: {sentence}")
    print()

    result = parse_sentence_to_amr(sentence, model, tokenizer, device)

    print(f"Status: {result['status']}")
    print(f"\nPredicted AMR:")
    print(result["amr_string"])
    print(f"\nGold AMR (linearized):")
    print(gold_amr)
    print()

## 7. Test: Custom Sentence

Try parsing your own sentence!

In [ ]:
# Try with a custom sentence
custom_sentence = "The boy wants to go to the park."

result = parse_sentence_to_amr(custom_sentence, model, tokenizer, device)

print(f"Input: {custom_sentence}")
print(f"Status: {result['status']}")
print(f"\nPredicted AMR:")
print(result["amr_string"])

## 8. Batch Parsing

Parse all test examples and summarize results.

In [ ]:
from common.postprocessing import ParsedStatus

print("Parsing all test examples...\n")

results = []
for i, example in enumerate(test_data):
    sentence = example["sent"]
    result = parse_sentence_to_amr(sentence, model, tokenizer, device)
    results.append(result)
    status_str = result["status"].name if hasattr(result["status"], "name") else str(result["status"])
    print(f"  [{i+1}/{len(test_data)}] {status_str:8s} | {sentence[:70]}...")

# Summary
ok_count = sum(1 for r in results if r["status"] == ParsedStatus.OK)
fixed_count = sum(1 for r in results if r["status"] == ParsedStatus.FIXED)
backoff_count = sum(1 for r in results if r["status"] == ParsedStatus.BACKOFF)

print(f"\n{'='*60}")
print(f"PARSING SUMMARY")
print(f"{'='*60}")
print(f"Total examples: {len(results)}")
print(f"  OK:      {ok_count} ({ok_count/len(results)*100:.1f}%)")
print(f"  FIXED:   {fixed_count} ({fixed_count/len(results)*100:.1f}%)")
print(f"  BACKOFF: {backoff_count} ({backoff_count/len(results)*100:.1f}%)")

## 9. Inspect a Detailed Result

Let's look at one result in detail, including the raw token IDs.

In [ ]:
# Pick the first test result
example = test_data[0]
result = results[0]

print(f"Sentence: {example['sent']}")
print(f"\nParsing status: {result['status']}")
print(f"Number of predicted tokens: {len(result['raw_pred_ids'])}")
print(f"\nDecoded tokens (raw):")
print(tokenizer.decode(result['raw_pred_ids'], skip_special_tokens=False))
print(f"\nPredicted AMR graph:")
print(result['amr_string'])
print(f"\nGold AMR (linearized):")
print(example['amr'])